In [4]:
import pandas as pd
import numpy as np
from pathlib import Path

# Show more columns/width so we can actually read the wide MCP tables.
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)

# Notebooks live in notebooks/, data lives one level up in data/raw/
RAW = Path("../data/raw")
PROCESSED = Path("../data/processed")

print("Raw folder exists:", RAW.exists())
print("Files found:", len(list(RAW.glob("*.csv"))))

Raw folder exists: False
Files found: 0


In [7]:
import pandas as pd
import numpy as np
from pathlib import Path

# Show more columns/width so we can actually read the wide MCP tables.
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)

# Locate the project root robustly. Depending on where the notebook was
# saved, the working directory may be the project root itself or the
# notebooks/ subfolder, so we check both instead of hardcoding "../".
CWD = Path.cwd()

if (CWD / "data" / "raw").exists():
    PROJECT_ROOT = CWD
elif (CWD.parent / "data" / "raw").exists():
    PROJECT_ROOT = CWD.parent
else:
    raise FileNotFoundError(
        f"Could not find data/raw. Notebook is running in: {CWD}"
    )

RAW = PROJECT_ROOT / "data" / "raw"
PROCESSED = PROJECT_ROOT / "data" / "processed"
FIGURES = PROJECT_ROOT / "figures"

# Create output folders if they do not exist yet.
PROCESSED.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)

print("Working directory:", CWD)
print("Project root:", PROJECT_ROOT)
print("CSV files found:", len(list(RAW.glob("*.csv"))))

Working directory: C:\Users\dovyd\Documents\tennis-shot-quality
Project root: C:\Users\dovyd\Documents\tennis-shot-quality
CSV files found: 6


In [8]:
# Match-level metadata: players, tournament, surface, round.
# Needed later for Target 10 (surface-specific performance).
matches = pd.concat([
    load_with_gender("charting-m-matches.csv", "M"),
    load_with_gender("charting-w-matches.csv", "W"),
], ignore_index=True)

print("Matches shape:", matches.shape)
print("\nColumns:")
print(list(matches.columns))
matches.head(3)

Matches shape: (11646, 16)

Columns:
['match_id', 'Player 1', 'Player 2', 'Pl 1 hand', 'Pl 2 hand', 'Date', 'Tournament', 'Round', 'Time', 'Court', 'Surface', 'Umpire', 'Best of', 'Final TB?', 'Charted by', 'gender']


,match_id,Player 1,Player 2,Pl 1 hand,Pl 2 hand,Date,Tournament,Round,Time,Court,Surface,Umpire,Best of,Final TB?,Charted by,gender
0,20260521-M-Roland_Garros-Q3-Jesper_De_Jong-Mic...,Jesper De Jong,Michael Zheng,R,R,20260521,Roland Garros,Q3,NaN,7,Clay,NaN,3,A,stard54,M
1,20260517-M-Rome_Masters-F-Casper_Ruud-Jannik_S...,Casper Ruud,Jannik Sinner,R,R,20260517,Rome Masters,F,5pm,Centre,Clay,Renaud Lichtenstein,3,1,Edo,M
2,20260511-M-Rome_Masters-R16-Rafael_Jodar-Learn...,Rafael Jodar,Learner Tien,R,L,20260511,Rome Masters,R16,NaN,NaN,Clay,NaN,3,1,stard54,M


In [9]:
# Point-by-point data. This is the core dataset - each row is one point,
# with the shot sequence stored as notation strings in the serve columns.
points = pd.concat([
    load_with_gender("charting-m-points-2010s.csv", "M"),
    load_with_gender("charting-m-points-2020s.csv", "M"),
    load_with_gender("charting-w-points-2010s.csv", "W"),
    load_with_gender("charting-w-points-2020s.csv", "W"),
], ignore_index=True)

print("Points shape:", points.shape)
print("\nColumns:")
print(list(points.columns))

Points shape: (1415633, 15)

Columns:
['match_id', 'Pt', 'Set1', 'Set2', 'Gm1', 'Gm2', 'Pts', 'Gm#', 'TbSet', 'Svr', '1st', '2nd', 'Notes', 'PtWinner', 'gender']


In [10]:
# Inspect dtypes and memory footprint before deciding on any cleaning.
points.info(memory_usage="deep")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1415633 entries, 0 to 1415632
Data columns (total 15 columns):
 #   Column    Non-Null Count    Dtype  
---  ------    --------------    -----  
 0   match_id  1415633 non-null  object 
 1   Pt        1415633 non-null  int64  
 2   Set1      1415633 non-null  int64  
 3   Set2      1415633 non-null  int64  
 4   Gm1       1415633 non-null  int64  
 5   Gm2       1415631 non-null  float64
 6   Pts       1415633 non-null  object 
 7   Gm#       1415632 non-null  float64
 8   TbSet     1415632 non-null  object 
 9   Svr       1415633 non-null  int64  
 10  1st       1415633 non-null  object 
 11  2nd       526311 non-null   object 
 12  Notes     112559 non-null   object 
 13  PtWinner  1415633 non-null  int64  
 14  gender    1415633 non-null  object 
dtypes: float64(2), int64(6), object(7)
memory usage: 598.0 MB


In [11]:
# Look at actual rows. The notation columns are the ones we will parse in 02.
points.head(5)

,match_id,Pt,Set1,Set2,Gm1,Gm2,Pts,Gm#,TbSet,Svr,1st,2nd,Notes,PtWinner,gender
0,20191124-M-Davis_Cup_Finals-F-Rafael_Nadal-Den...,1,0,0,0,0.0,0-0,1.0,True,1,4n,4b1b1b3n@,NaN,1,M
1,20191124-M-Davis_Cup_Finals-F-Rafael_Nadal-Den...,2,0,0,0,0.0,15-0,1.0,True,1,5n,5f28f2b1b1b2f3f3d@,NaN,1,M
2,20191124-M-Davis_Cup_Finals-F-Rafael_Nadal-Den...,3,0,0,0,0.0,30-0,1.0,True,1,4C,NaN,NaN,1,M
3,20191124-M-Davis_Cup_Finals-F-Rafael_Nadal-Den...,4,0,0,0,0.0,40-0,1.0,True,1,4r2d#,NaN,NaN,1,M
4,20191124-M-Davis_Cup_Finals-F-Rafael_Nadal-Den...,5,0,0,1,0.0,0-0,2.0,True,2,5d,c4b1b1d!@,NaN,1,M


In [12]:
# Share of missing values per column, highest first.
# Some columns are legitimately sparse (e.g. second serve only exists
# when the first serve was a fault), so this is descriptive, not a to-do list.
missing = points.isna().mean().sort_values(ascending=False) * 100
missing[missing > 0].round(1)

Notes    92.0
2nd      62.8
Gm2       0.0
Gm#       0.0
TbSet     0.0
dtype: float64

In [13]:
# Sanity check that the concatenation worked and both sources are present.
print(points["gender"].value_counts())
print("\nUnique matches:", points["match_id"].nunique())
print("Unique matches by gender:")
print(points.groupby("gender")["match_id"].nunique())

gender
M    904707
W    510926
Name: count, dtype: int64

Unique matches: 9247
Unique matches by gender:
gender
M    5569
W    3678
Name: match_id, dtype: int64


In [14]:
# Check how usable the Surface column is before committing to Target 10.
# If coverage is poor we would need to infer surface from tournament name.
print("Surface missing (%):", round(matches["Surface"].isna().mean() * 100, 1))
print("\nSurface value counts:")
print(matches["Surface"].value_counts(dropna=False))

Surface missing (%): 0.0

Surface value counts:
Surface
Hard                  7600
Clay                  2752
Grass                 1283
1                        7
Eva Asderaki-Moore       2
3                        2
Name: count, dtype: int64


In [15]:
# First look at the raw notation strings we will need to parse in notebook 02.
# Each string encodes the full point: serve, then every rally shot, then
# a terminating symbol indicating how the point ended.
sample = points[points["1st"].str.len() > 6].head(15)
for _, row in sample.iterrows():
    print(f"{row['1st']:<40} | PtWinner: {row['PtWinner']}")

4f38f+3f1i^1i3n#                         | PtWinner: 2
4r17b1m3o3*                              | PtWinner: 2
4b28f3f1*                                | PtWinner: 2
4f28b+1b3v3*                             | PtWinner: 1
5s27f+3*                                 | PtWinner: 1
6f37f+3*                                 | PtWinner: 2
6f37f+3f1z^1*                            | PtWinner: 2
4f38f1*                                  | PtWinner: 2
4f28f1*                                  | PtWinner: 2
5f18f3f1b1w@                             | PtWinner: 1
c5b28f1b1b2f2d@                          | PtWinner: 1
4+f2z2b3n#                               | PtWinner: 1
6f18f1f1x@                               | PtWinner: 2
4f39f2f3f3f1d@                           | PtWinner: 1
c6f19b1w#                                | PtWinner: 1


In [16]:
# Save the merged frames so later notebooks do not need to re-read
# and re-concatenate 140 MB of csv every time.
points.to_parquet(PROCESSED / "points_raw.parquet", index=False)
matches.to_parquet(PROCESSED / "matches_raw.parquet", index=False)

print("Saved to:", PROCESSED)

Saved to: C:\Users\dovyd\Documents\tennis-shot-quality\data\processed
